# Haja Coração - Pipeline BPM otimizado

Esta versão calcula as mesmas métricas da baseline, mas aplica otimizações manuais controladas:

- selecionar e filtrar antes do join;
- usar `broadcast` na dimensão pequena de dispositivos;
- não usar `repartition()` ou `coalesce()` sem necessidade;
- manter AQE e broadcast automático desligados para medir o efeito das otimizações manuais;
- usar a mesma ação final da baseline para comparar tempo, stages e DAG.

A entrada é o XLSX gerado automaticamente pelo `gerador_batimentos.py`. O XLSX é convertido para CSV somente para ingestão; o tratamento é feito pelo Spark.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, avg, stddev, min, max, count, sum as spark_sum, when, round as spark_round
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
import pandas as pd
import os
import time

spark = (SparkSession.builder
    .appName("HajaCoracao-BPM-Otimizado")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "6")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast automatico:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("Spark UI: http://localhost:4040")


ModuleNotFoundError: No module named 'pyspark'

## 1. Leitura do XLSX

O Spark SQL não lê XLSX com `spark.read.csv/parquet`. Por isso o notebook usa pandas **somente na entrada**, convertendo a planilha para CSV. Em um cenário maior, a origem ideal seria CSV/JSON/Parquet ou outro formato distribuído.


In [ ]:
XLSX = "/home/ubuntu/dados_batimentos.xlsx"
CSV = "/home/ubuntu/dados_batimentos.csv"

if not os.path.exists(CSV):
    pd.read_excel(XLSX, sheet_name="Dados Batimentos").to_csv(CSV, index=False)

schema = StructType([
    StructField("messageId", IntegerType(), False),
    StructField("deviceId", StringType(), False),
    StructField("heartRate", DoubleType(), False),
    StructField("heartRateTarget", DoubleType(), False),
    StructField("activityState", IntegerType(), False),
    StructField("activityLabel", StringType(), False),
    StructField("bpmAlert", BooleanType(), False),
    StructField("timestamp", StringType(), False),
    StructField("deviceIndex", IntegerType(), False),
    StructField("timeSinceStart", StringType(), False)
])

bpm = spark.read.option("header", True).schema(schema).csv(CSV)
bpm.printSchema()
bpm.show(5, truncate=False)


## 2. Otimização manual: projeção e filtro antes do join

O `select` reduz as colunas que entram nas etapas seguintes. Os filtros eliminam registros inválidos o mais cedo possível.

Isso segue a lógica de **Projection Pruning** e **Predicate Pushdown** apresentada no material: reduzir dados antes de operações caras.


In [ ]:
inicio = time.time()

# Otimizacao 1: selecionar e filtrar antes do join.
# Menos colunas e menos registros chegam ao shuffle da agregacao.
bpm_filtrado = (bpm
    .select("deviceId", "heartRate", "activityLabel", "bpmAlert", "timestamp")
    .filter(col("heartRate").isNotNull())
    .filter((col("heartRate") >= 30) & (col("heartRate") <= 220))
 )

print("Registros apos filtro:", bpm_filtrado.count())
bpm_filtrado.show(5, truncate=False)

## 3. Broadcast: dimensão pequena de dispositivos

Para demonstrar a técnica sem inventar um cenário artificial de tabela grande, modelamos os metadados dos 6 dispositivos como uma pequena dimensão.

`broadcast(dim_devices)` envia essa tabela pequena aos executors e permite que o lado grande faça o join sem o shuffle que seria necessário em um join baseado em redistribuição.

Com apenas 500 registros, o ganho de tempo será pequeno ou até imperceptível; o objetivo é demonstrar a estratégia que escala para uma tabela BPM grande + dimensão pequena.


In [ ]:
dim_devices = spark.createDataFrame([
    ("device-01",), ("device-02",), ("device-03",),
    ("device-04",), ("device-05",), ("device-06",)
], ["deviceId"])

# Otimizacao 2: a dimensao de dispositivos e pequena.
# O broadcast manual evita o shuffle dessa tabela no join.
bpm_completo = bpm_filtrado.join(
    broadcast(dim_devices),
    on="deviceId",
    how="inner"
 )

bpm_completo.explain("formatted")

## 4. Agregação e ação comparável

O `groupBy` é uma operação **wide** e normalmente gera `Exchange`, pois registros da mesma chave podem estar em partições diferentes.

A ação final usa `orderBy(...).show()` nos dois notebooks para que a comparação de tempo seja justa. O `orderBy` também pode gerar `Exchange`, mas existe apenas para ordenar a amostra exibida.

AQE e broadcast automático permanecem desligados; o ganho observado deve vir do filtro/projeção antecipados e do `broadcast` manual.

In [ ]:
# Otimizacao 3: nao usamos repartition, orderBy ou distinct sem necessidade.
# O unico shuffle esperado e o Exchange necessario ao groupBy.
resultado = (bpm_completo
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
    .groupBy("deviceId", "activityLabel")
    .agg(
        count("*").alias("total_registros"),
        spark_round(avg("heartRate"), 2).alias("media_bpm"),
        spark_round(stddev("heartRate"), 2).alias("desvio_bpm"),
        min("heartRate").alias("min_bpm"),
        max("heartRate").alias("max_bpm"),
        spark_sum(when(col("bpmAlert") == True, 1).otherwise(0)).alias("alertas")
    )
 )

resultado.explain("formatted")
resultado.orderBy("deviceId", "activityLabel").show(20, truncate=False)

fim = time.time()
print(f"Tempo total de execucao: {fim - inicio:.2f} segundos")

## 5. Plano físico e geração de código

Use o `formatted` para registrar `Exchange`, `BroadcastExchange` e a estratégia de join. Use `codegen` para observar a geração de código pelo Spark.

A comparação conceitual é:
**código PySpark → plano lógico → plano otimizado → plano físico → execução/codegen**.


In [ ]:
print("=== OPERACOES WIDE DA VERSAO OTIMIZADA ===")
print("join: broadcast(dim_devices) gera BroadcastExchange, evitando shuffle da dimensao.")
print("groupBy: gera Exchange; e necessario para reunir as chaves da agregacao.")
print("orderBy: gera Exchange adicional, usado apenas para uma comparacao justa da acao final.")
print("distinct/repartition/coalesce: nao usados.")
print("Filtro e select: aplicados antes do join para reduzir dados movimentados.")
print("\n=== PLANO FISICO ===")
resultado.explain("formatted")
print("\n=== CONFIGURACOES ===")
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast automatico:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI: http://localhost:4040")

## 6. Resultado tratado

Parquet é usado como saída porque preserva tipos e é adequado para leitura colunar pelo Spark.

A ordenação é feita apenas para apresentação. Ela é uma operação potencialmente wide; portanto, não é necessária para a geração do resultado analítico e não deve ser adicionada ao pipeline apenas para “organizar” os dados.


In [ ]:
OUTPUT = "/home/ubuntu/processed_bpm_parquet"

resultado.write.mode("overwrite").parquet(OUTPUT)
print("Saida:", OUTPUT)
print("Spark UI ativa em http://localhost:4041")
print("Nao execute spark.stop() antes de consultar Jobs, Stages e DAG.")

## 7. Evidências para o relatório

Na EC2, acesse a Spark UI por túnel SSH em `http://localhost:4040`.

Na UI:
1. **Jobs** → abra o job gerado pela ação;
2. capture **DAG Visualization**;
3. abra **Stages**;
4. registre tempo, número de tasks, Shuffle Read e Shuffle Write;
5. no `explain("formatted")`, procure `Exchange` e `BroadcastExchange`.

Não invente métricas: os valores do relatório devem ser os observados na execução.


In [ ]:
print("Spark UI continua disponível enquanto esta SparkSession estiver ativa.")
print("Não execute spark.stop() se quiser explorar a UI durante a sessão.")
